# Estimated intrinsic dimension across the toy-manifold noise sweep

Reproduce the summary table and per-manifold mean-rank heatmap from
`toy_manifold_noise_sweep_results.ipynb` using the completed **2NN and GRIDE ID estimates**.
Training and independent test populations remain separate. **Execute this notebook in Slurm.**

- ID was estimated in the original 128-dimensional coordinates using each model's native assignments.
- A cluster contributes only with **at least 500 points in that split and a valid estimate**.
- A cluster's manifold is the unique nearest noiseless manifold to its fitted centre, using the original evaluator and saved distance cutoff. The same association is used for train and test.
- Each heatmap cell is an **unweighted mean over valid clusters**; `n` is the number of contributors.
- The summary table gives **equal weight to each manifold with an available mean**. It records contributing manifold and cluster counts; missing manifolds are never treated as zero.
- GRIDE's six neighbour-rank pairs stay separate. No averaging across scales or rounding to integer ranks is performed.
- Colours reproduce the original plot: absolute distance between the **mean estimate** and the planted **noiseless** intrinsic dimension `r`. This reference does not assert that noisy data have ID `r` at every scale.
- These eligibility-filtered means use a different component population from the original learned-rank table. They also need not use the same population in train and test.

The notebook reads saved estimates and projects eligible centres; it does not rerun an ID estimator or fit a model.

In [ ]:
from pathlib import Path
import json
import os
import sys

assert os.environ.get("SLURM_JOB_ID"), "Run the notebook in a Slurm allocation, not on a login node."
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "src/dalg").is_dir(), "Start from the repository root or notebooks/."
sys.path[:0] = [str(REPO_ROOT / "src"), str(REPO_ROOT / "scripts/temporary")]

import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from plot_cluster_id_experiment import (
    ESTIMATORS, GRIDE_RANKS, MODEL_NAMES, build_manifold_tables, macro_summary, select_runs,
)
%matplotlib inline

ID_ROOT = REPO_ROOT / "outputs/experiments/id_est_exp/full_20261009_01"
MANIFEST = REPO_ROOT / "outputs/experiments/id_est_exp/validation_20261009_02/manifest.json"
ARTIFACT_DIR = Path(os.environ.get(
    "ID_FIGURE_OUTPUT",
    REPO_ROOT / "outputs/experiments/id_est_exp/figures_20261009_01",
))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": 10, "figure.facecolor": "white", "savefig.facecolor": "white"})

## Select the fitted runs

Initial selections match the original notebook: **K=2000**, all four noise conditions,
and **threshold 0.05** for KMeans, HDDC Adam, and HDDC EM.
All 740 completed runs remain available through the selectors.

`em` means shared noise variance; `em_bk` means component-specific noise variance.
They are distinct configurations. Vanilla MFA has no rank-selection threshold.
Changing the selection requires rerunning the preparation cell in Slurm.

In [ ]:
K = 2000
HDDC_THRESHOLD = 0.05
EM_THRESHOLD = 0.05
KMEANS_THRESHOLD = 0.05
EM_BK_THRESHOLD = 0.05
NOISE_LEVELS = ["noiseless", 1000, 100, 10]
MODELS = ["kmeans", "hddc", "em"]  # Also available: "mfa", "em_bk".
SPLITS = ["train", "test"]
ID_DISTANCE_COLOR_MAX = 5
ID_DISTANCE_COLOR_GAMMA = 0.5

thresholds = {"kmeans": KMEANS_THRESHOLD, "hddc": HDDC_THRESHOLD,
              "em": EM_THRESHOLD, "em_bk": EM_BK_THRESHOLD}
assert NOISE_LEVELS and MODELS and len(MODELS) == len(set(MODELS))
assert set(MODELS) <= MODEL_NAMES.keys()
plan = json.loads(MANIFEST.read_text())
selected = select_runs(plan, K=K, thresholds=thresholds,
                       noise_levels=NOISE_LEVELS, models=MODELS)
display(pd.DataFrame([{
    "Noise ratio": item["noise_ratio"], "Model": MODEL_NAMES[item["model"]],
    "K": item["run"]["K"], "Threshold": item["surgery_threshold"],
    "Run": item["run"]["relative_run"],
} for item in selected]))
selection = {"K": K, "thresholds": thresholds, "noise_levels": NOISE_LEVELS,
             "models": MODELS, "splits": SPLITS, "gride_ranks": list(GRIDE_RANKS),
             "id_root": str(ID_ROOT), "manifest": str(MANIFEST),
             "slurm_job_id": os.environ["SLURM_JOB_ID"],
             "relative_runs": [item["run"]["relative_run"] for item in selected]}
(ARTIFACT_DIR / "selection.json").write_text(json.dumps(selection, indent=2) + "\n")

## Prepare the manifold summaries

The saved per-cluster estimates are checked against their completion fingerprints.
Only centres of clusters eligible in at least one split need geometric projection.
Original per-manifold metrics supply the total associated-component counts.

This cell writes `manifold_ids.csv`, `macro_ids.csv`,
`eligible_component_manifolds.csv`, and `sources.json` under the figure output directory.

In [ ]:
manifold_ids = build_manifold_tables(plan, selected, ID_ROOT, ARTIFACT_DIR)
macro_ids = macro_summary(manifold_ids)
macro_ids.to_csv(ARTIFACT_DIR / "macro_ids.csv", index=False)
manifold_info = (
    manifold_ids[["type_name", "intrinsic_dim", "embedding_dim"]].drop_duplicates()
    .sort_values(["intrinsic_dim", "embedding_dim"], kind="stable")
    .set_index("type_name")
)
assert manifold_info.index.is_unique and len(manifold_info) == 10
column_order = pd.MultiIndex.from_product(
    [NOISE_LEVELS, MODELS], names=["noise_ratio", "model"],
)
split_names = {"train": "Training", "test": "Independent test"}
settings_label = f"K={K}; " + "; ".join(
    f"{MODEL_NAMES[model]} threshold={thresholds[model]:g}"
    for model in MODELS if model != "mfa"
)
print(f"Prepared {len(selected)} fitted runs, {len(manifold_ids)} manifold/split rows.")
print(settings_label)

def save_figure(fig, name):
    for extension in ("png", "pdf"):
        fig.savefig(ARTIFACT_DIR / f"{name}.{extension}", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## Summary tables: mean estimated ID

Each entry shows **mean ID**, followed by **contributing manifolds / 10; n=contributing clusters**.
This is a mean of the available manifold means, not a pooled or point-weighted cluster mean.
Different entries can have different contributors; a dash denotes an undefined mean.

In [ ]:
summary_tables = {}
for split in SPLITS:
    rows = macro_ids.loc[macro_ids["split"].eq(split)]
    table = pd.DataFrame(index=column_order)
    table["Noise ratio"] = [str(noise) for noise, model in column_order]
    table["Model"] = [MODEL_NAMES[model] for noise, model in column_order]
    for estimator, label in ESTIMATORS.items():
        values = rows.loc[rows["estimator"].eq(estimator)].set_index(
            ["noise_ratio", "model"]
        ).reindex(column_order)
        cells = []
        for _, row in values.iterrows():
            if pd.isna(row["total_manifolds"]):
                cells.append("—")
            else:
                value = "—" if pd.isna(row["mean_id"]) else f"{row['mean_id']:.2f}"
                cells.append(
                    f"{value}\n{int(row['valid_manifolds'])}/{int(row['total_manifolds'])}; "
                    f"n={int(row['valid_clusters'])}"
                )
        table[label] = cells
    summary_tables[split] = table.reset_index(drop=True)
    display(summary_tables[split].style.set_caption(
        f"{split_names[split]} — equal-manifold mean estimated ID"
    ).set_properties(**{"white-space": "pre-line"}))
    fig, ax = plt.subplots(figsize=(16, 1.6 + 0.54 * len(table)), layout="constrained")
    ax.axis("off")
    artist = ax.table(
        cellText=table.to_numpy(), colLabels=table.columns,
        colWidths=[0.065, 0.165] + [0.11] * len(ESTIMATORS),
        cellLoc="center", loc="center",
    )
    artist.auto_set_font_size(False)
    artist.set_fontsize(9)
    for (row, column), cell in artist.get_celld().items():
        cell.set_edgecolor("#d7dee6")
        cell.set_height(0.94 / (len(table) + 1))
        if row == 0:
            cell.set_facecolor("#20344a")
            cell.set_text_props(color="white", weight="bold")
        else:
            cell.set_facecolor("#eef3f8" if ((row - 1) // len(MODELS)) % 2 == 0 else "white")
    ax.set_title(f"{split_names[split]} — mean estimated intrinsic dimension", loc="left",
                 fontsize=16, fontweight="bold", pad=16)
    fig.supxlabel(
        settings_label + "\nEach cell: mean ID; contributing manifolds/10 and cluster count n. "
        "Equal weight per available manifold; clusters require ≥500 points.",
        fontsize=9,
    )
    save_figure(fig, f"summary_{split}")

## Per-manifold heatmaps

Rows and noise/model column groups follow the original notebook.
Each cell shows the **mean ID** and its **valid cluster count n**.
Colour is `|mean ID − r|`, using the original power scale and a cap of 5;
the printed ID is never clipped. Gray cells have no available mean.

Training and independent test are separate panels with the same colour scale.

In [ ]:
def plot_id_heatmaps(estimator):
    row_labels = [
        f"{name.replace('_', ' ')} ({int(row.intrinsic_dim)},{int(row.embedding_dim)})"
        for name, row in manifold_info.iterrows()
    ]
    short_names = {"kmeans": "KMeans", "mfa": "MFA", "hddc": "HDDC\nAdam",
                   "em": "EM\nshared b", "em_bk": "EM\ncomponent b"}
    fig, axes = plt.subplots(
        len(SPLITS), 1, squeeze=False,
        figsize=(max(10, 3.2 + 0.83 * len(column_order)), 1.4 + 6.4 * len(SPLITS)),
        layout="constrained", facecolor="white",
    )
    for split, ax in zip(SPLITS, axes.flat):
        rows = manifold_ids.loc[manifold_ids["split"].eq(split)]
        means = rows.pivot(index="type_name", columns=["noise_ratio", "model"],
                           values=f"{estimator}_mean").reindex(
                               index=manifold_info.index, columns=column_order)
        counts = rows.pivot(index="type_name", columns=["noise_ratio", "model"],
                            values=f"{estimator}_n").reindex(
                                index=manifold_info.index, columns=column_order)
        distance = means.sub(manifold_info["intrinsic_dim"], axis="index").abs()
        annotations = np.full(means.shape, "", dtype=object)
        for row in range(len(means)):
            for col in range(len(means.columns)):
                value, count = means.iat[row, col], counts.iat[row, col]
                if pd.notna(value):
                    annotations[row, col] = f"{value:.2f}\nn={int(count)}"
        sns.heatmap(
            distance, ax=ax, annot=annotations, fmt="", cmap="viridis_r",
            vmin=0, vmax=ID_DISTANCE_COLOR_MAX,
            norm=PowerNorm(gamma=ID_DISTANCE_COLOR_GAMMA, vmin=0,
                           vmax=ID_DISTANCE_COLOR_MAX, clip=True),
            linewidths=0.5, linecolor="white", annot_kws={"fontsize": 9},
            xticklabels=[short_names[model] for _, model in column_order],
            yticklabels=row_labels,
            cbar_kws={"label": "Distance from noiseless r: |mean ID − r|", "shrink": 0.9},
        )
        bar = ax.collections[0].colorbar
        ticks = np.linspace(0, ID_DISTANCE_COLOR_MAX, 6)
        bar.set_ticks(ticks)
        bar.set_ticklabels([f"{value:g}" for value in ticks[:-1]] + [f"{ticks[-1]:g}+"])
        ax.set_facecolor("#e9edf2")
        for row, col in np.argwhere(means.isna().to_numpy()):
            count = counts.iat[row, col]
            label = "—" if pd.isna(count) else f"—\nn={int(count)}"
            ax.text(col + 0.5, row + 0.5, label, ha="center", va="center",
                    color="#626b78", fontsize=9)
        ax.xaxis.tick_top()
        ax.tick_params(axis="both", length=0, labelrotation=0, labelsize=9)
        ax.set(xlabel="", ylabel="")
        ax.set_title(split_names[split], loc="left", fontsize=13, fontweight="bold", pad=68)
        ax.text(-0.02, 1.12, "Noise ratio", transform=ax.transAxes,
                ha="right", va="center", fontsize=9)
        for group, noise in enumerate(NOISE_LEVELS):
            ax.text(group * len(MODELS) + len(MODELS) / 2, 1.12, str(noise),
                    transform=ax.get_xaxis_transform(), ha="center", va="center", weight="bold")
            if group:
                ax.axvline(group * len(MODELS), color="white", linewidth=3)
        low_dim_rows = manifold_info["intrinsic_dim"].le(2).sum()
        if 0 < low_dim_rows < len(means):
            ax.axhline(low_dim_rows, color="white", linewidth=3)
    fig.suptitle(f"Mean estimated intrinsic dimension — {ESTIMATORS[estimator]}",
                 fontsize=17, fontweight="bold")
    fig.supxlabel(
        settings_label + "\nRows: manifold (r,m). Labels: mean ID and valid clusters n (≥500 points each)."
        f"\nColour: |mean ID − r|, power scale capped at {ID_DISTANCE_COLOR_MAX:g}. "
        "r is the planted noiseless reference; gray — means unavailable.",
        fontsize=9,
    )
    save_figure(fig, estimator)

In [ ]:
plot_id_heatmaps("two_nn")

## GRIDE: all six scales

Each figure uses the indicated neighbour-rank pair `(n1, n2)`, where `n2 = 2 n1`.
Equal ranks do not imply equal physical distance across clusters or splits.
The original results retain those distance scales and the reported estimator errors.

In [ ]:
for rank in GRIDE_RANKS:
    plot_id_heatmaps(f"gride_{rank}")

## Exported artifacts

The output directory contains the two rendered summary tables and seven figures
(2NN and six GRIDE scales), each in PNG and PDF, plus the numerical CSVs and source fingerprints.
The Slurm launcher also saves an executed copy of this notebook.

In [ ]:
print(ARTIFACT_DIR)
for path in sorted(ARTIFACT_DIR.glob("*.png")):
    print(path.name)